In [1]:
!pip install pypdf sentence-transformers faiss-cpu transformers


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pypdf import PdfReader
reader=PdfReader(r"C:\Users\Meghna\Downloads\Historical-underpinnings.pdf")
text=""
for page in reader.pages:
    text+=page.extract_text()

In [3]:
documents=[]
chunk_size=500
for i in range(0,len(text),chunk_size):
    documents.append(text[i:i+chunk_size])
    print(len(documents))

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24


In [4]:
from sentence_transformers import SentenceTransformer
model=SentenceTransformer("all-MiniLM-L6-v2")
embeddings=model.encode(documents)

In [5]:
import faiss
import numpy as np
dimension=embeddings.shape[1]
index=faiss.IndexFlatL2(dimension)
index.add(embeddings.astype("float32"))

In [6]:
from transformers import pipeline
generator=pipeline("text2text-generation",model="google/flan-t5-large")

Device set to use cpu


In [ ]:
while True:
    print("\n1 Summary")
    print("2 Questions")
    print("3 Exit")
    task=input("Choose : ")
    if task=="3":
        break
    if task=="1":
        context=""
        for i in range (min(2,len(documents))):
            context+=documents[i]+"\n"
            context=context[:1000]
        prompt=f"""
        Give a short and concise summary in points of the text:

        Context:
        {context}
        """
        answer=generator(prompt,max_new_tokens=150)
        print(answer[0]["generated_text"])
        
        
    elif task=="2":   
        
        
        question=input("Ask question: ")
        query_embedding=model.encode([question])
        distances,indices=index.search(query_embedding.astype("float32"),k=1)
        
        context=""
        for i in indices[0]:
            context+=documents[i]
        
        prompt=f"""
           

        Context:
        {context}

        Question:
        {question}
        """
        answer=generator(prompt,max_length=300)
        print(answer[0]["generated_text"])
              
    


1 Summary
2 Questions
3 Exit


Choose :  2
Ask question:  When was jhansi annexed by doctrine of lapse.


Both `max_new_tokens` (=256) and `max_length`(=300) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1861

1 Summary
2 Questions
3 Exit
